# Klasyfikacja osobników manula (Manul Re-Identification)
### Projekt na zajęcia z sieci neuronowych
**Autor:** Zespół projektowy  
**Zadanie:** Rozpoznawanie konkretnych osobników manula (*Bol, Grusha, Nagomu, Primula, Prinsessa*) na podstawie wykadrowanych zdjęć z portalu manulization.com.

---
### Plan eksperymentów (zgodnie z instrukcją sylabusową):
1. **Przygotowanie danych:** Podział na zbiory Train / Val / Test (70% / 15% / 15%), augmentacja danych.
2. **Badana architektura sieci (Custom CNN):**
   - Wpływ współczynnika uczenia (*Learning Rate*): `0.01`, `0.001`, `0.0001`
   - Wpływ funkcji aktywacji: `ReLU`, `LeakyReLU`, `GELU`
   - Wpływ liczby warstw ukrytych i filtrów splotowych: 2, 3 oraz 4 bloki konwolucyjne
   - Wpływ liczby neuronów w warstwie gęstej: 128, 256, 512
3. **Proces uczenia i ewaluacja:**
   - Algorytmy: Adam oraz SGD z momentum
   - Warunek zatrzymania: Early Stopping (monitorowanie `val_loss`)
   - Kryteria błędu: Cross-Entropy Loss oraz MSE Loss (pomocniczo wg instrukcji)
4. **Wizualizacje i raport:** Wykresy zbieżności, Confusion Matrix, tabela metryk.
5. **Transfer Learning:** Porównanie z pretrenowanym modelem ResNet-50.


## 1. Środowisko i rozpakowanie danych
Przeciągnij plik `manul_dataset.zip` na lewy pasek plików w Colabie (ikona folderu), a następnie uruchom poniższą komórkę.


In [ ]:
import os
import torch
import zipfile

# 1. Sprawdzenie GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Wykorzystywane urządzenie: {device}")
if torch.cuda.is_available():
    print(f"Karta graficzna: {torch.cuda.get_device_name(0)}")

# 2. Weryfikacja obecności i poprawności pliku zip
zip_path = "manul_dataset.zip"
if not os.path.exists(zip_path) and os.path.exists("/content/manul_dataset.zip"):
    zip_path = "/content/manul_dataset.zip"

is_valid_zip = False
if os.path.exists(zip_path):
    size_mb = os.path.getsize(zip_path) / (1024 * 1024)
    print(f"Znaleziono plik {zip_path} ({size_mb:.2f} MB).")
    try:
        with zipfile.ZipFile(zip_path, "r") as zf:
            if zf.testzip() is None:
                is_valid_zip = True
                print("Plik ZIP jest kompletny i poprawny!")
    except Exception as e:
        print(f"Plik ZIP jest niekompletny (jeszcze się wgrywa lub przerwano): {e}")

# Jeśli plik nie istnieje lub jest niekompletny, prosimy o wgranie przez okienko
if not is_valid_zip and not (os.path.exists("/content/data/cropped") or os.path.exists("data/cropped")):
    print("\nKliknij przycisk poniżej i wskaż plik manul_dataset.zip z dysku:")
    from google.colab import files
    files.upload()
    zip_path = "manul_dataset.zip"

# 3. Bezpieczne rozpakowanie w Pythonie
if not (os.path.exists("/content/data/cropped") or os.path.exists("data/cropped")):
    print("\nRozpakowywanie archiwum...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall("/content")
    print("Pomyślnie rozpakowano zbiór danych!")
else:
    print("Folder data/cropped już istnieje.")

# 4. Podsumowanie klas
DATA_DIR = "/content/data/cropped" if os.path.exists("/content/data/cropped") else "data/cropped"
classes = sorted(os.listdir(DATA_DIR))
print(f"\nLiczba klas: {len(classes)} -> {classes}")
for c in classes:
    cnt = len(os.listdir(os.path.join(DATA_DIR, c)))
    print(f"  - {c:12s}: {cnt} wyciętych zdjęć")


## 2. Podział na zbiory (Train, Val, Test) oraz Augmentacja Danych
Stosujemy stratyfikowany podział: **70% zbiór treningowy**, **15% zbiór walidacyjny**, **15% zbiór testowy**.


In [ ]:
import random
import numpy as np
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

# Ustalenie ziarna losowości dla powtarzalności wyników
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Transformacje dla zbioru treningowego (Augmentacja zapobiegająca przeuczeniu)
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.85, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=12),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Transformacje dla zbioru walidacyjnego i testowego (tylko normalizacja bez augmentacji)
eval_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Wczytanie pełnego zbioru danych
base_dataset = datasets.ImageFolder(DATA_DIR)
class_names = base_dataset.classes
num_classes = len(class_names)
total_samples = len(base_dataset)

# Stratyfikowany podział indeksów
from collections import defaultdict
class_indices = defaultdict(list)
for idx, (_, label) in enumerate(base_dataset.samples):
    class_indices[label].append(idx)

train_indices, val_indices, test_indices = [], [], []

for label, indices in class_indices.items():
    random.shuffle(indices)
    n = len(indices)
    n_train = int(0.70 * n)
    n_val = int(0.15 * n)
    train_indices.extend(indices[:n_train])
    val_indices.extend(indices[n_train:n_train + n_val])
    test_indices.extend(indices[n_train + n_val:])

# Dedykowane datasety z odpowiednimi transformacjami
train_dataset = Subset(datasets.ImageFolder(DATA_DIR, transform=train_transforms), train_indices)
val_dataset = Subset(datasets.ImageFolder(DATA_DIR, transform=eval_transforms), val_indices)
test_dataset = Subset(datasets.ImageFolder(DATA_DIR, transform=eval_transforms), test_indices)

BATCH_SIZE = 16
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Liczba próbek ogółem: {total_samples}")
print(f"  - Treningowe:  {len(train_indices)} ({len(train_indices)/total_samples:.1%})")
print(f"  - Walidacyjne: {len(val_indices)} ({len(val_indices)/total_samples:.1%})")
print(f"  - Testowe:     {len(test_indices)} ({len(test_indices)/total_samples:.1%})")


### Podgląd przykładowych wykadrowanych manuli


In [ ]:
def imshow_batch(images, labels, classes):
    images = images.numpy().transpose((0, 2, 3, 1))
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    images = std * images + mean
    images = np.clip(images, 0, 1)

    fig = plt.figure(figsize=(12, 6))
    for idx in range(min(8, len(images))):
        ax = fig.add_subplot(2, 4, idx + 1, xticks=[], yticks=[])
        plt.imshow(images[idx])
        ax.set_title(classes[labels[idx]], fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.show()

dataiter = iter(train_loader)
images, labels = next(dataiter)
imshow_batch(images, labels, class_names)


## 3. Modułowa Architektura Sieci Neuronowej (`CustomCNN`)
Zgodnie z wymaganiami uczelnianymi, klasa sieci pozwala na pełną parametryzację:
- Liczby bloków konwolucyjnych i filtrów (np. `[32, 64]`, `[32, 64, 128]`, `[32, 64, 128, 256]`)
- Liczby neuronów w warstwie gęstej (`fc_units`)
- Funkcji aktywacji (`ReLU`, `LeakyReLU`, `GELU`)
- Dropoutu i Batch Normalization


In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class CustomCNN(nn.Module):
    def __init__(self, conv_channels=[32, 64, 128], fc_units=256, activation_fn=nn.ReLU, dropout_rate=0.4, num_classes=5):
        super(CustomCNN, self).__init__()
        
        layers = []
        in_c = 3
        for out_c in conv_channels:
            layers.append(nn.Conv2d(in_c, out_c, kernel_size=3, padding=1))
            layers.append(nn.BatchNorm2d(out_c))
            layers.append(activation_fn())
            layers.append(nn.MaxPool2d(kernel_size=2, stride=2))
            in_c = out_c
            
        self.features = nn.Sequential(*layers)
        self.adaptive_pool = nn.AdaptiveAvgPool2d((4, 4))
        
        flat_features = in_c * 4 * 4
        self.classifier = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(flat_features, fc_units),
            nn.BatchNorm1d(fc_units),
            activation_fn(),
            nn.Dropout(dropout_rate),
            nn.Linear(fc_units, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.adaptive_pool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

# Testowy przebieg w celu weryfikacji wymiarów tensorów
test_model = CustomCNN().to(device)
dummy_input = torch.randn(2, 3, 224, 224).to(device)
dummy_out = test_model(dummy_input)
print(f"Struktura modelu zdefiniowana poprawnie! Kształt wyjścia: {dummy_out.shape}")


## 4. Pętla Trenująca, Early Stopping i Kryteria Błędu (Cross-Entropy + MSE)
Wymagania formalne:
- Implementacja algorytmu uczenia się (SGD / Adam)
- Warunek zatrzymania: Early Stopping
- Kryterium pomiaru błędu: Cross-Entropy Loss oraz pomocniczo błąd średniokwadratowy MSE


In [ ]:
import copy
from sklearn.metrics import accuracy_score, f1_score

def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=40, patience=8, device=device):
    best_model_wts = copy.deepcopy(model.state_dict())
    best_val_loss = float("inf")
    patience_counter = 0

    history = {
        "train_loss": [], "val_loss": [],
        "train_mse": [], "val_mse": [],
        "train_acc": [], "val_acc": []
    }
    
    mse_criterion = nn.MSELoss()

    for epoch in range(1, num_epochs + 1):
        # --- FAZA TRENINGOWA ---
        model.train()
        running_loss = 0.0
        running_mse = 0.0
        all_preds = []
        all_targets = []

        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()

            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)

            # Pomocnicze obliczenie błędu MSE na one-hot probabilities (wymóg sylabusa)
            with torch.no_grad():
                probs = F.softmax(outputs, dim=1)
                one_hot_targets = F.one_hot(targets, num_classes=num_classes).float()
                mse = mse_criterion(probs, one_hot_targets)
                running_mse += mse.item() * inputs.size(0)

            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(targets.cpu().numpy())

        epoch_train_loss = running_loss / len(train_loader.dataset)
        epoch_train_mse = running_mse / len(train_loader.dataset)
        epoch_train_acc = accuracy_score(all_targets, all_preds)

        # --- FAZA WALIDACYJNA ---
        model.eval()
        running_val_loss = 0.0
        running_val_mse = 0.0
        val_preds = []
        val_targets = []

        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, targets)

                running_val_loss += loss.item() * inputs.size(0)
                probs = F.softmax(outputs, dim=1)
                one_hot_targets = F.one_hot(targets, num_classes=num_classes).float()
                running_val_mse += mse_criterion(probs, one_hot_targets).item() * inputs.size(0)

                _, preds = torch.max(outputs, 1)
                val_preds.extend(preds.cpu().numpy())
                val_targets.extend(targets.cpu().numpy())

        epoch_val_loss = running_val_loss / len(val_loader.dataset)
        epoch_val_mse = running_val_mse / len(val_loader.dataset)
        epoch_val_acc = accuracy_score(val_targets, val_preds)

        history["train_loss"].append(epoch_train_loss)
        history["val_loss"].append(epoch_val_loss)
        history["train_mse"].append(epoch_train_mse)
        history["val_mse"].append(epoch_val_mse)
        history["train_acc"].append(epoch_train_acc)
        history["val_acc"].append(epoch_val_acc)

        if epoch % 5 == 0 or epoch == 1:
            print(f"Epoka {epoch:02d}/{num_epochs:02d} | "
                  f"Train Loss: {epoch_train_loss:.4f} (Acc: {epoch_train_acc:.1%}) | "
                  f"Val Loss: {epoch_val_loss:.4f} (Acc: {epoch_val_acc:.1%})")

        # Early Stopping: zatrzymanie przy braku poprawy val_loss
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            best_model_wts = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"-> Early stopping w epoce {epoch}! Najlepszy Val Loss: {best_val_loss:.4f}")
                break

    model.load_state_dict(best_model_wts)
    return model, history

def evaluate_on_test(model, test_loader, criterion, device=device):
    model.eval()
    running_loss = 0.0
    running_mse = 0.0
    all_preds = []
    all_targets = []
    mse_criterion = nn.MSELoss()

    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            running_loss += loss.item() * inputs.size(0)

            probs = F.softmax(outputs, dim=1)
            one_hot_targets = F.one_hot(targets, num_classes=num_classes).float()
            running_mse += mse_criterion(probs, one_hot_targets).item() * inputs.size(0)

            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(targets.cpu().numpy())

    test_loss = running_loss / len(test_loader.dataset)
    test_mse = running_mse / len(test_loader.dataset)
    test_acc = accuracy_score(all_targets, all_preds)
    test_f1 = f1_score(all_targets, all_preds, average="weighted")
    return test_loss, test_mse, test_acc, test_f1, all_preds, all_targets


## 5. Seria Eksperymentów (Badane Hiperparametry i Architektura)
Zgodnie z sylabusem przeprowadzamy serię eksperymentów badających wpływ:
1. **Współczynnika uczenia (*Learning Rate*):** `0.01`, `0.001`, `0.0001`
2. **Funkcji aktywacji:** `ReLU`, `LeakyReLU`, `GELU`
3. **Liczby warstw ukrytych i filtrów splotowych:**
   - Model A (2 bloki): `[32, 64]`
   - Model B (3 bloki): `[32, 64, 128]`
   - Model C (4 bloki): `[32, 64, 128, 256]`
4. **Liczby neuronów w warstwie gęstej (FC):** `128` vs `256` vs `512`


In [ ]:
experiments_results = {}
criterion = nn.CrossEntropyLoss()

# --- EKSPERYMENT 1: Wpływ Learning Rate ---
print("==================================================")
print("EKSPERYMENT 1: Wpływ Learning Rate (lr = 0.01, 0.001, 0.0001)")
print("==================================================")

lr_list = [0.01, 0.001, 0.0001]
for lr in lr_list:
    name = f"LR_{lr}"
    print(f"\n--- Trening: {name} ---")
    model = CustomCNN(conv_channels=[32, 64, 128], fc_units=256, activation_fn=nn.ReLU).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    trained_model, hist = train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=35, patience=7)
    test_loss, test_mse, test_acc, test_f1, _, _ = evaluate_on_test(trained_model, test_loader, criterion)
    
    experiments_results[name] = {
        "history": hist,
        "test_acc": test_acc,
        "test_f1": test_f1,
        "test_loss": test_loss,
        "test_mse": test_mse,
        "model": trained_model
    }
    print(f"Wynik testowy {name}: Acc = {test_acc:.1%}, F1 = {test_f1:.3f}, Loss = {test_loss:.4f}, MSE = {test_mse:.4f}")


In [ ]:
# --- EKSPERYMENT 2: Wpływ Funkcji Aktywacji ---
print("==================================================")
print("EKSPERYMENT 2: Funkcje Aktywacji (ReLU, LeakyReLU, GELU)")
print("==================================================")

activations = {
    "Act_ReLU": nn.ReLU,
    "Act_LeakyReLU": nn.LeakyReLU,
    "Act_GELU": nn.GELU
}

for act_name, act_fn in activations.items():
    if act_name in experiments_results:
        continue
    print(f"\n--- Trening: {act_name} ---")
    model = CustomCNN(conv_channels=[32, 64, 128], fc_units=256, activation_fn=act_fn).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    
    trained_model, hist = train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=35, patience=7)
    test_loss, test_mse, test_acc, test_f1, _, _ = evaluate_on_test(trained_model, test_loader, criterion)
    
    experiments_results[act_name] = {
        "history": hist,
        "test_acc": test_acc,
        "test_f1": test_f1,
        "test_loss": test_loss,
        "test_mse": test_mse,
        "model": trained_model
    }
    print(f"Wynik testowy {act_name}: Acc = {test_acc:.1%}, F1 = {test_f1:.3f}")


In [ ]:
# --- EKSPERYMENT 3: Głębokość sieci i filtry konwolucyjne ---
print("==================================================")
print("EKSPERYMENT 3: Liczba warstw splotowych i filtrów")
print("==================================================")

depth_configs = {
    "Conv_2_Layers_[32,64]": [32, 64],
    "Conv_3_Layers_[32,64,128]": [32, 64, 128],
    "Conv_4_Layers_[32,64,128,256]": [32, 64, 128, 256]
}

for cfg_name, channels in depth_configs.items():
    if cfg_name in experiments_results:
        continue
    print(f"\n--- Trening: {cfg_name} ---")
    model = CustomCNN(conv_channels=channels, fc_units=256, activation_fn=nn.ReLU).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    
    trained_model, hist = train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=35, patience=7)
    test_loss, test_mse, test_acc, test_f1, _, _ = evaluate_on_test(trained_model, test_loader, criterion)
    
    experiments_results[cfg_name] = {
        "history": hist,
        "test_acc": test_acc,
        "test_f1": test_f1,
        "test_loss": test_loss,
        "test_mse": test_mse,
        "model": trained_model
    }
    print(f"Wynik testowy {cfg_name}: Acc = {test_acc:.1%}, F1 = {test_f1:.3f}")


## 6. Zestawienie Wyników Eksperymentów (Tabela i Wykresy do Sprawozdania)


In [ ]:
import pandas as pd

summary_data = []
for name, data in experiments_results.items():
    summary_data.append({
        "Eksperyment / Konfiguracja": name,
        "Test Accuracy": f"{data['test_acc']:.2%}",
        "Test F1-Score": f"{data['test_f1']:.3f}",
        "Test Cross-Entropy Loss": f"{data['test_loss']:.4f}",
        "Test MSE Loss": f"{data['test_mse']:.4f}",
        "Epok przed Early Stop": len(data["history"]["val_loss"])
    })

df_results = pd.DataFrame(summary_data)
display(df_results)

# Wykresy porównawcze dla Learning Rate
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for lr in [0.01, 0.001, 0.0001]:
    key = f"LR_{lr}"
    if key in experiments_results:
        hist = experiments_results[key]["history"]
        axes[0].plot(hist["val_loss"], label=f"Val Loss (lr={lr})")
        axes[1].plot(hist["val_acc"], label=f"Val Acc (lr={lr})")

axes[0].set_title("Wpływ Learning Rate na Funkcję Straty (Val Loss)")
axes[0].set_xlabel("Epoka")
axes[0].set_ylabel("Cross-Entropy Loss")
axes[0].grid(True)
axes[0].legend()

axes[1].set_title("Wpływ Learning Rate na Dokładność (Val Accuracy)")
axes[1].set_xlabel("Epoka")
axes[1].set_ylabel("Accuracy")
axes[1].grid(True)
axes[1].legend()

plt.tight_layout()
plt.savefig("lr_comparison.png", dpi=300)
plt.show()


## 7. Macierz Pomyłek (Confusion Matrix) Najlepszego Modelu
Niezbędna do sprawozdania w celu analizy, które manule są do siebie najbardziej podobne wizualnie.


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

# Wybór najlepszego modelu
best_exp_name = max(experiments_results, key=lambda k: experiments_results[k]["test_acc"])
best_exp = experiments_results[best_exp_name]
best_model = best_exp["model"]

print(f"Najlepszy model: {best_exp_name} (Dokładność testowa: {best_exp['test_acc']:.1%})")

# Ewaluacja i predykcje
_, _, _, _, test_preds, test_targets = evaluate_on_test(best_model, test_loader, criterion)

print("\n--- Raport Klasyfikacji ---")
print(classification_report(test_targets, test_preds, target_names=class_names))

cm = confusion_matrix(test_targets, test_preds)
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.title(f"Macierz Pomyłek - {best_exp_name}", fontsize=14, fontweight="bold")
plt.xlabel("Przewidziany manul", fontsize=12)
plt.ylabel("Rzeczywisty manul", fontsize=12)
plt.tight_layout()
plt.savefig("confusion_matrix_custom_cnn.png", dpi=300)
plt.show()


## 8. Transfer Learning (Fine-Tuning ResNet-50)
Etap 2 z sylabusa: Porównanie własnej sieci trenowanej od zera z modelem pretrenowanym na ImageNet.


In [ ]:
from torchvision.models import resnet50, ResNet50_Weights

print("Pobieranie wag pretrenowanego modelu ResNet-50...")
weights = ResNet50_Weights.DEFAULT
resnet = resnet50(weights=weights)

# Zamiana ostatniej warstwy FC na 5 klas
in_features = resnet.fc.in_features
resnet.fc = nn.Sequential(
    nn.Dropout(0.4),
    nn.Linear(in_features, num_classes)
)
resnet = resnet.to(device)

optimizer_resnet = torch.optim.Adam(resnet.parameters(), lr=0.0001)

print("\nRozpoczynam Fine-Tuning ResNet-50...")
resnet_trained, resnet_hist = train_model(
    resnet, train_loader, val_loader, criterion, optimizer_resnet, num_epochs=20, patience=5
)

r_loss, r_mse, r_acc, r_f1, r_preds, r_targets = evaluate_on_test(resnet_trained, test_loader, criterion)

print(f"\n==========================================")
print(f"WYNIKI TRANSFER LEARNING (ResNet-50):")
print(f"  - Test Accuracy: {r_acc:.1%}")
print(f"  - Test F1-Score: {r_f1:.3f}")
print(f"  - Test Cross-Entropy: {r_loss:.4f}")
print(f"  - Test MSE: {r_mse:.4f}")
print(f"==========================================")

# Porównanie z własną siecią Custom CNN
print(f"\nPORÓWNANIE DO SPRAWOZDANIA:")
print(f"  Custom CNN ({best_exp_name}): {best_exp['test_acc']:.1%}")
print(f"  ResNet-50 Fine-Tuning:         {r_acc:.1%}")
